# Getting Started with Strands Agents

## Overview
Strands Agents is a powerful framework for building AI agents that can interact with AWS services and perform complex tasks. This notebook walks you through creating your first Strands agent, from a simple agent to one that uses custom tools, and finishes with a practical RecipeBot use case.

## Tutorial Details

| Information          | Details                              |
|:---------------------|:-------------------------------------|
| Agent structure      | Single agent                         |
| Model provider       | Amazon Bedrock                       |
| Model                | Anthropic Claude Sonnet 4.5          |
| Native tools used    | none                                 |
| Custom tools created | calculator, weather, websearch |
| Strands features     | Agent, @tool decorator, BedrockModel |

## What you'll learn
* Create an agent and give it a system prompt
* Add custom tools
* Invoke a tool directly with `agent.tool.<name>`
* Configure logging
* Choose and configure a model provider

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)

Let's now install the required packages for our agent.

In [1]:
# Install Strands using pip

!pip install strands-agents


## Creating Your First Agent

Let's get an overview of the agentic components needed.

### Create a simple agent

This will create an agent with the default model provider, [Amazon Bedrock](https://aws.amazon.com/bedrock/). If you do not pass a `model`, the agent uses the default Amazon Bedrock model (covered in the Model provider section below). While the agent runs in your local environment, the Amazon Bedrock model runs in the cloud, and your agent invokes it there. The architecture looks as follows:

<div style="text-align:center">
    <img src="images/simple_agent.png" width="75%" />
</div>

In [3]:
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow") 

from strands import Agent
# Initialize your agent
agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    system_prompt="You are a helpful assistant that provides concise responses."
)

# Send a message to the agent
response = agent("Hello! Tell me a joke.")

Hello! Here's a joke for you:

Why did the scarecrow win an award?

Because he was outstanding in his field! 🌾

### Add tools to the agent

You create custom tools using the `@tool` decorator. The [strands-agents-tools](https://github.com/strands-agents/tools) repository also provides pre-built tools that you can import. The function's typed arguments and docstring are not just documentation. Strands turns them into the tool's input schema, which is what the model reads to decide when and how to call the tool. We can create agents with custom tools. For instance, defining a small calculator tool and a custom tool for getting the weather, you get the following architecture:
<div style="text-align:center">
    <img src="images/agent_with_tools.png" width="75%" />
</div>

Implementing this architecture, you have the following:

In [4]:
import operator
from typing import Literal

from strands import Agent, tool

_OPS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
    "**": operator.pow,
}


@tool
def calculator(a: float, b: float, op: Literal["+", "-", "*", "/", "**"]) -> float:
    """Apply an arithmetic operator to two numbers.

    Args:
        a: Left operand.
        b: Right operand.
        op: One of "+", "-", "*", "/", "**".
    """
    return _OPS[op](a, b)

# Create a custom tool
@tool
def weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: Name of the city to get the weather for.
    """
    # Dummy implementation - a real tool would call a weather API
    return f"It is sunny and 72°F in {city}."

agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather.")

response = agent("What is the weather in Seattle today?")
print(response)


Tool #1: weather
The weather in Seattle today is sunny and 72°F. It's a beautiful day!The weather in Seattle today is sunny and 72°F. It's a beautiful day!



### Invoking tool directly

For some applications it is important to directly call the tool. For instance, you might want to debug the tool, pre-populate the agent knowledge with your customer's information, or use a tool inside another tool. In Strands you can do it using the ``tool`` method of your agent followed by the tool name.

In [5]:
# Alternatively, you can invoke the tool directly like so:
agent.tool.calculator(a=144, b=0.5, op="**")

{'toolUseId': 'tooluse_calculator_454415806',
 'status': 'success',
 'content': [{'text': '12.0'}]}


### Change the log level and format

Strands Agents SDK uses Python's standard `logging` module to provide visibility into its operations.

The Strands Agents SDK implements a straightforward logging approach:

1. **Module-level Loggers**: Each module in the SDK creates its own logger using logging.getLogger(__name__), following Python best practices for hierarchical logging.
2. **Root Logger**: All loggers in the SDK are children of the "strands" root logger, making it easy to configure logging for the entire SDK.
3. **Default Behavior**: By default, the SDK doesn't configure any handlers or log levels, allowing you to integrate it with your application's logging configuration.

To enable logging for the Strands Agents SDK, you can configure the **"strands"** logger. If you want to change the log level, for example during debugging, or modify the log format, you can set the logger configuration as follows:

In [6]:
import logging
from strands import Agent

# Enables Strands debug log level
logging.getLogger("strands").setLevel(logging.DEBUG) # or logging.INFO

# Sets the logging format and streams logs to stderr
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

agent = Agent(model="us.anthropic.claude-sonnet-4-5-20250929-v1:0")
agent("Hello!")

DEBUG | strands.models.bedrock | config=<{'model_id': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'}> | initializing
DEBUG | strands.models.bedrock | region=<us-west-2> | bedrock client created
DEBUG | strands.tools.registry | tools_dir=</home/sagemaker-user/agentic-workload-hec/01-fundamentals/lab-01-getting-started/tools> | tools directory not found
DEBUG | strands.tools.registry | tool_modules=<[]> | discovered
DEBUG | strands.tools.registry | tool_count=<0>, success_count=<0> | finished loading tools
DEBUG | strands.tools.registry | getting tool configurations
DEBUG | strands.tools.registry | tool_count=<0> | tools configured
DEBUG | strands.tools.registry | getting tool configurations
DEBUG | strands.tools.registry | tool_count=<0> | tools configured
DEBUG | strands.event_loop.streaming | model=<<strands.models.bedrock.BedrockModel object at 0x7fbcf915af30>> | streaming messages
DEBUG | strands.models.bedrock | formatting request
DEBUG | strands.models.bedrock | request=<{'modelI

Hello! How can I help you today?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Hello! How can I help you today?'}]}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[3.629960536956787], traces=[<strands.telemetry.metrics.Trace object at 0x7fbcf915adb0>], accumulated_usage={'inputTokens': 9, 'outputTokens': 12, 'totalTokens': 21}, accumulated_metrics={'latencyMs': 3593}), state={})


### Model provider

The default model provider is [Amazon Bedrock](https://aws.amazon.com/bedrock/). If you do not pass a `model`, the agent uses the SDK's default Bedrock model. Your AWS region selects the Bedrock endpoint your agent calls, not which default model it gets.

You can specify a different model in Amazon Bedrock providing the model ID string directly:

In [7]:
from strands import Agent

agent = Agent(model="us.anthropic.claude-haiku-4-5-20251001-v1:0")
print(agent.model.config)

DEBUG | strands.models.bedrock | config=<{'model_id': 'us.anthropic.claude-haiku-4-5-20251001-v1:0'}> | initializing
DEBUG | strands.models.bedrock | region=<us-west-2> | bedrock client created
DEBUG | strands.tools.registry | tools_dir=</home/sagemaker-user/agentic-workload-hec/01-fundamentals/lab-01-getting-started/tools> | tools directory not found
DEBUG | strands.tools.registry | tool_modules=<[]> | discovered
DEBUG | strands.tools.registry | tool_count=<0>, success_count=<0> | finished loading tools


{'model_id': 'us.anthropic.claude-haiku-4-5-20251001-v1:0'}



For more control over the model configuration, you can create a `BedrockModel` provider instance:

In [8]:
import boto3
from strands import Agent
from strands.models import BedrockModel

# Create a BedrockModel
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name='us-west-2',
    temperature=0.3,
)

agent = Agent(model=bedrock_model)

DEBUG | strands.models.bedrock | config=<{'model_id': 'us.anthropic.claude-haiku-4-5-20251001-v1:0', 'temperature': 0.3}> | initializing
DEBUG | strands.models.bedrock | region=<us-west-2> | bedrock client created
DEBUG | strands.tools.registry | tools_dir=</home/sagemaker-user/agentic-workload-hec/01-fundamentals/lab-01-getting-started/tools> | tools directory not found
DEBUG | strands.tools.registry | tool_modules=<[]> | discovered
DEBUG | strands.tools.registry | tool_count=<0>, success_count=<0> | finished loading tools



More details on the available model providers on the [Model Provider Quickstart page](https://strandsagents.com/docs/user-guide/quickstart/python/#model-providers)


## Let's Build a Task-Specific Agent - RecipeBot

Let's create a practical example of a task-specific agent. We create a `RecipeBot` that recommends recipes and answers any cooking-related questions.

Here's what we will create:

<div style="text-align:center">
    <img src="images/interactive_recipe_agent.png" width="75%" />
</div>

In [9]:
# Install ddgs for the RecipeBot web search tool
%pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 131.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ddgs]
Note: you may need to restart the kernel to use updated packages.


In [10]:
from strands import Agent, tool
from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException
import logging

# Configure logging
logging.getLogger("strands").setLevel(logging.INFO)

# Define a websearch tool
@tool
def websearch(keywords: str, region: str = "us-en", max_results: int | None = None) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"


In [11]:
# Create a recipe assistant agent
recipe_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""You are RecipeBot, a helpful cooking assistant.
    Help users find recipes based on ingredients and answer cooking questions.
    Use the websearch tool to find recipes when users mention ingredients or to
    look up cooking information.""",
    # Import the websearch tool we created above
    tools=[websearch],
)

In [12]:
response = recipe_agent("Suggest a recipe with chicken and broccoli.")

# Other examples:
# response = recipe_agent("How do I cook quinoa?")
# response = recipe_agent("How can I substitute white wine in shrimp pasta?")
# response = recipe_agent("What are the health benefits of asparagus?")
print(response)


Tool #1: websearch
Let me try that search again:
Tool #2: websearch
Great! Here are some wonderful chicken and broccoli recipe options for you:

## **Easy Chicken and Broccoli Stir-Fry** (5 Ingredients!)
This is a quick and delicious option:
- **Ingredients**: Chicken pieces, fresh broccoli, soy sauce, ginger, garlic, and brown sugar
- **Method**: Stir-fry chicken pieces with fresh broccoli in a mix of these ingredients
- **Serve with**: Rice for a complete, easy dinner
- **Time**: About 30 minutes

## **30-Minute Chicken and Broccoli Casserole**
A comforting option:
- This creamy, cheesy comfort food uses simple ingredients with big flavor
- Made with chicken breast, frozen broccoli, cream of mushroom soup, cheese, and instant rice
- Perfect for a weeknight when you want something cozy and filling

## **One-Pot Chicken and Broccoli Skillet**
A Mediterranean-inspired version:
- Features bright flavors of garlic, lemon, and oregano
- Sweet and savory sauce that beats takeout
- All made

For more detail, check out the [Strands documentation](https://strandsagents.com/docs/user-guide/quickstart/python/).

## Congratulations!

In this notebook you built your first Strands agent, added custom tools, invoked tools directly, configured logging, chose a model provider, and assembled a RecipeBot. Next, let's look at adding tools and connecting MCP servers.

---

# Exercise — A paper-finding agent

**Time: 10 minutes.**

You have seen RecipeBot search the web with a `websearch` tool. Now build your own: an agent that
finds **arXiv papers** on a topic you give it, and answers with real links.

The catch: a plain web search for a machine-learning topic returns blog posts, tutorials and
newsletters. You want arXiv and nothing else. A language model asked politely to "only use arXiv"
will drift. So the constraint has to live **inside your tool**, not in a hope.

**Write two things:**

1. A `@tool` that searches the web but only returns results from `arxiv.org`.
   Look at the `websearch` tool above for the `DDGS` call and its error handling.
   The search-engine syntax to scope a query to one domain is `site:<domain>`.
2. A `system_prompt` for an agent that uses your tool and always shows the paper links.

Then run the self-check: every link in the answer must be an arXiv link.

In [16]:
import json
import re
import warnings

warnings.filterwarnings(action="ignore")

from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException
from strands import Agent, tool

# Records what your tool returned, so the self-check can inspect it.
CALLS = []

TOPIC = "retrieval augmented generation"
QUESTION = f"Find me 3 recent papers about {TOPIC}. Show the link for each one."

In [20]:
# ---------------------------------------------------------------------------
# YOUR WORK GOES HERE
#
# 1. Write the tool. It must return web results that are all from arxiv.org.
#    Remember the docstring and type hints ARE the schema the model reads.
# 2. Write the system prompt.
# ---------------------------------------------------------------------------

@tool
def find_papers(topic: str, max_results: int = 5) -> str:
    """TODO: replace this docstring. Describe what the tool returns and what each argument means."""
    ## TODO: build a query that restricts the search to arxiv.org, run it with DDGS,
    """
    Search the web for papers on arxiv.org about a given topic.

    Args:
        topic: The research topic to search for.
        max_results: The maximum number of arXiv results to return.

    Returns:
        A JSON string containing search results from arxiv.org only.
    """
    query = f"site:arxiv.org {topic}"

    try:
        results = list(
            DDGS().text(
                query,
                max_results=max_results
            )
        )

        # Extra protection: remove anything not from arxiv.org
        results = [
            r for r in results
            if "arxiv.org" in r.get("href", "")
        ]

        CALLS.append(results)

        return json.dumps(results, ensure_ascii=False)

    except RatelimitException:
        return "RatelimitException: Please try again after a short delay"

    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"

    except Exception as e:
        return f"Exception: {e}"
    ## then: CALLS.append(results) before returning them.
    raise NotImplementedError("Write your tool, then re-run this cell.")


SYSTEM_PROMPT = """
You are an academic paper-finding agent.

When the user asks you to find papers:
- Always use the find_papers tool.
- Only recommend papers returned by the tool.
- Never invent paper titles, authors, or URLs.
- Only include papers hosted on arxiv.org.
- Always show the arXiv link for every paper you mention.
- If fewer papers are found than requested, return only the papers actually found.
"""

paper_agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=SYSTEM_PROMPT,
    tools=[find_papers],   # TODO: update if you renamed your tool
)

CALLS.clear()
answer = paper_agent(QUESTION)


Tool #1: find_papers
Let me try that search again:
Tool #2: find_papers
Here are 3 recent papers about retrieval augmented generation:

1. **Retrieval-Augmented Generation: A Comprehensive Survey**
   https://arxiv.org/abs/2506.00054
   This survey consolidates current knowledge in RAG research and serves as a foundation for the next generation of retrieval-augmented language modeling systems.

2. **PAR2-RAG: Planned Active Retrieval and Reasoning**
   https://arxiv.org/abs/2603.29085
   This paper addresses challenges in multi-hop settings where RAG approaches often fail when evidence coverage is incomplete or reasoning trajectories drift.

3. **Retrieval-Augmented Generation for Large Language Models**
   https://arxiv.org/abs/2312.10997
   This work explores how RAG enhances accuracy and credibility of generation for knowledge-intensive tasks by incorporating knowledge from external databases.

In [ ]:
links = re.findall(r"https?://[^\s)\]\>,]+", str(answer))
arxiv = [u for u in links if "arxiv.org" in u]

checks = {
    "your tool was actually called":
        len(CALLS) > 0,
    "the answer contains at least 3 links":
        len(links) >= 3,
    "every link in the answer is on arxiv.org":
        len(links) > 0 and len(arxiv) == len(links),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

print(f"\nlinks found: {len(links)}, of which on arxiv.org: {len(arxiv)}")
for u in links:
    print(("  ok   " if "arxiv.org" in u else "  NOT ARXIV  ") + u)

print()
print("Exercise complete." if all(checks.values())
      else "Not there yet - where is the arxiv.org constraint enforced?")

## Stuck?

One possible solution is stored below, encoded so you do not read it by accident. Try the exercise
first: the point is the attempt, not the answer.

Run `show_solution()` to print it, or `load_solution()` to run it in place of your own cell.

In [18]:
import base64

_SOLUTION = (
    "QHRvb2wKZGVmIGZpbmRfcGFwZXJzKHRvcGljOiBzdHIsIG1heF9yZXN1bHRzOiBpbnQgPSA1KSAtPiBzdHI6CiAgICAi"
    "IiJGaW5kIHJlY2VudCBhclhpdiBwYXBlcnMgYWJvdXQgYSByZXNlYXJjaCB0b3BpYy4KCiAgICBBcmdzOgogICAgICAg"
    "IHRvcGljOiBUaGUgcmVzZWFyY2ggdG9waWMgdG8gc2VhcmNoIGZvciwgZS5nLiAicmV0cmlldmFsIGF1Z21lbnRlZCBn"
    "ZW5lcmF0aW9uIi4KICAgICAgICBtYXhfcmVzdWx0czogSG93IG1hbnkgcGFwZXJzIHRvIHJldHVybi4KICAgICIiIgog"
    "ICAgdHJ5OgogICAgICAgIHJhdyA9IERER1MoKS50ZXh0KGYic2l0ZTphcnhpdi5vcmcge3RvcGljfSIsIHJlZ2lvbj0i"
    "dXMtZW4iLCBtYXhfcmVzdWx0cz1tYXhfcmVzdWx0cykKICAgIGV4Y2VwdCBSYXRlbGltaXRFeGNlcHRpb246CiAgICAg"
    "ICAgcmV0dXJuICJSYXRlbGltaXRFeGNlcHRpb246IHBsZWFzZSByZXRyeSBzaG9ydGx5LiIKICAgIGV4Y2VwdCBEREdT"
    "RXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmV0dXJuIGYiRHVja0R1Y2tHb1NlYXJjaEV4Y2VwdGlvbjoge2V9IgogICAg"
    "cGFwZXJzID0gW3IgZm9yIHIgaW4gKHJhdyBvciBbXSkgaWYgImFyeGl2Lm9yZyIgaW4gci5nZXQoImhyZWYiLCAiIild"
    "CiAgICBDQUxMUy5hcHBlbmQocGFwZXJzKQogICAgcmV0dXJuIGpzb24uZHVtcHMocGFwZXJzKSBpZiBwYXBlcnMgZWxz"
    "ZSAiTm8gcmVzdWx0cyBmb3VuZC4iCgoKU1lTVEVNX1BST01QVCA9ICIiIllvdSBhcmUgUGFwZXJCb3QuIFlvdSBmaW5k"
    "IGFjYWRlbWljIHBhcGVycyBvbiBhclhpdi4KQWx3YXlzIHVzZSB0aGUgZmluZF9wYXBlcnMgdG9vbDsgbmV2ZXIgYW5z"
    "d2VyIGZyb20gbWVtb3J5LgpMaXN0IGVhY2ggcGFwZXIgd2l0aCBpdHMgdGl0bGUgYW5kIGl0cyBleGFjdCBhcnhpdi5v"
    "cmcgbGluayBmcm9tIHRoZSB0b29sIHJlc3VsdHMuCk5ldmVyIGludmVudCBhIGxpbmsgYW5kIG5ldmVyIGNpdGUgYW55"
    "IHNpdGUgb3RoZXIgdGhhbiBhcnhpdi5vcmcuIiIiCgpwYXBlcl9hZ2VudCA9IEFnZW50KAogICAgbW9kZWw9InVzLmFu"
    "dGhyb3BpYy5jbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOS12MTowIiwKICAgIHN5c3RlbV9wcm9tcHQ9U1lTVEVNX1BS"
    "T01QVCwKICAgIHRvb2xzPVtmaW5kX3BhcGVyc10sCikKCkNBTExTLmNsZWFyKCkKYW5zd2VyID0gcGFwZXJfYWdlbnQo"
    "UVVFU1RJT04pCgojIE5vdGUgd2hlcmUgdGhlIGNvbnN0cmFpbnQgbGl2ZXM6ICJzaXRlOmFyeGl2Lm9yZyIgaXMgYmFr"
    "ZWQgaW50byB0aGUgcXVlcnksIGFuZCB0aGUKIyByZXN1bHRzIGFyZSBmaWx0ZXJlZCBhZ2FpbiBvbiB0aGUgd2F5IG91"
    "dC4gVGhlIHByb21wdCByZWluZm9yY2VzIGl0IGJ1dCBuZXZlcgojIGVuZm9yY2VzIGl0IG9uIGl0cyBvd24u"
)


def show_solution() -> None:
    """Print one possible solution to the exercise."""
    print(base64.b64decode(_SOLUTION).decode())


def load_solution() -> None:
    """Run the reference solution, then re-run the self-check cell to see it pass."""
    exec(base64.b64decode(_SOLUTION).decode(), globals())


print("Ready. Call show_solution() to read it, or load_solution() to run it.")

Ready. Call show_solution() to read it, or load_solution() to run it.
